## 0 · Goal & config
TODO: goal, model, seed, paths.

In [5]:
# Runpod only: skip this cell when running locally.
%cd /workspace/LLM-fine-tuning/Training

[Errno 2] No such file or directory: '/workspace/LLM-fine-tuning/Training'
/Users/faanjum/Developer/LLM-fine-tuning/Training


### Set device

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")

print("device-name", device);                      

device-name mps


#### tokenizer

In [2]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

## 1 · Load model
Load pretrained GPT-2 and move it to the device.

In [3]:
from reference.configs import gpt2_BASE_CONFIG, gpt2_model_configs

CHOOSE_MODEL = "gpt2-medium (355M)" # gpt2-small (124M), gpt2-medium (355M), gpt2-large (774M), gpt2-xl (1558M)

# ** unpacks dictionary entries: {**a, **b} combines them; b wins on duplicate keys.
model_config = {**gpt2_BASE_CONFIG, **gpt2_model_configs[CHOOSE_MODEL]}


In [9]:
%%capture
from reference.model_io import download_weights
from reference.models.gpt2 import GPT2Model, load_weights_into_gpt

# download_weights() defaults to 124M
w = download_weights("openai-community/gpt2-medium")
model = GPT2Model(model_config)
load_weights_into_gpt(model, w)

model = model.to(device)
model.eval()

## 2 · Data + DataLoader  ← differs
Format prompts, split, dataset, collate (see masking note), loaders.

In [4]:
from reference.data import load_instruction_data

data = load_instruction_data()
print("Instruction Data Keys: ", data[0].keys())
data[:1]

Instruction Data Keys:  dict_keys(['instruction', 'input', 'output'])


[{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
  'input': 'freind --> friend',
  'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}]

### Split data: train 0.85, validation 0.05, test 0.10

In [5]:
train_portion = int(0.85 * len(data))
test_portion  = int(0.10 * len(data))
val_portion   = int(len(data) - train_portion - test_portion)

train_data = data[:train_portion]
test_data  = data[train_portion:train_portion+test_portion]
val_data   = data[train_portion+test_portion:]


print(f"shape: {train_data[0].keys()}")
print(f"train Length: {len(train_data)}")
print(f"test Length: {len(test_data)}")
print(f"val Length: {len(val_data)}")

shape: dict_keys(['instruction', 'input', 'output'])
train Length: 935
test Length: 110
val Length: 55


### Prompt template

- Combines an example dictionary's instruction and optional input into one text prompt. The response is added separately

In [6]:
def format_input(entry):

  instruction_text = (
      f"Below is an instruction that describes a task. "
      f"Write a response that appropriately completes the request."
      f"\n\n### Instruction:\n{entry['instruction']}"
  )

  input_text = (
  f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""
  )

  return instruction_text + input_text

model_input = format_input(data[50])
detailed_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + detailed_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


### Dataset
Dataset returns token IDs after **formatting + encoding**; collate_fn creates the inputs and shifted targets


In [13]:
from torch.utils.data import Dataset

class InstructionDataset(Dataset):

    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            formatted_input    = format_input(entry)
            formatted_response = f"\n\n### Response:\n{entry['output']}"
            combined_text = formatted_input + formatted_response

            #  → list: [10, 20, 30] as custom_collate_fn expects lists for .copy() and padding
            encoded_text = tokenizer.encode(combined_text)
            self.encoded_texts.append(encoded_text)


    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)


# Example
dataset = InstructionDataset(train_data, tokenizer)
print(f"Training Dataset Length: {len(dataset)} -> same as train_data")
print(f"dataset[0] ({len(dataset[0])} tokens): {dataset[0]}")
print(f"dataset[1] ({len(dataset[1])} tokens): {dataset[1]}")

Training Dataset Length: 935 -> same as train_data
dataset[0] (74 tokens): [21106, 318, 281, 12064, 326, 8477, 257, 4876, 13, 19430, 257, 2882, 326, 20431, 32543, 262, 2581, 13, 198, 198, 21017, 46486, 25, 198, 36, 2100, 4985, 262, 1708, 9546, 416, 25449, 340, 656, 262, 24993, 1813, 13, 198, 198, 21017, 23412, 25, 198, 19503, 521, 14610, 1545, 198, 198, 21017, 18261, 25, 198, 464, 24993, 286, 262, 1813, 9546, 366, 19503, 521, 1, 318, 11491, 11, 262, 3376, 24993, 318, 366, 6726, 1911]
dataset[1] (59 tokens): [21106, 318, 281, 12064, 326, 8477, 257, 4876, 13, 19430, 257, 2882, 326, 20431, 32543, 262, 2581, 13, 198, 198, 21017, 46486, 25, 198, 18378, 262, 1708, 6827, 329, 23491, 13, 198, 198, 21017, 23412, 25, 198, 1544, 467, 284, 262, 3952, 790, 1110, 13, 198, 198, 21017, 18261, 25, 198, 1544, 2925, 284, 262, 3952, 790, 1110, 13]


### Custom Collate Function

**note**: Pretraining notebook run moves batches to the device inside the training loop via calc_loss_batch; here, custom_collate_fn moves them during batch preparation

In [14]:
import torch

def custom_collate_fn(batch, pad_token_id, ignore_index = -100, device = "cpu", allowed_max_length = None):

    batch_max_length = max(len(item)+ 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs  = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        # mask = torch.tensor([False, False, True, True])
        mask    = targets == pad_token_id

        # torch.nonzero(mask) tensor([[2], [3]]) --> squeeze() tensor([2, 3])
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            # targets[torch.tensor([3])] = ignore_index
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs  = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # This moves the tensors to the device
    inputs_tensor  = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor


# Example - 2 examples in batch
batch = [dataset[0], dataset[1]] # expects list of examples
input_batch, target_batch = custom_collate_fn(batch, pad_token_id=tokenizer.eot_token)
print(f"Same lengths within ANY batch after padding:")
print(f"Each input:  {input_batch.shape[1]} tokens")
print(f"Each target: {target_batch.shape[1]} tokens\n")

# Example - Inputs versus targets
print(f"inputs[0]:  {input_batch[0][:5].tolist()} ... {input_batch[0][-5:].tolist()}")
print(f"targets[0]: {target_batch[0][:5].tolist()} ...  {target_batch[0][-5:].tolist()}\n")

print(f"Input tokens in entire batch:  {input_batch.numel()}")
print(f"Target tokens in entire batch: {target_batch.numel()}")


Same lengths within ANY batch after padding:
Each input:  74 tokens
Each target: 74 tokens

inputs[0]:  [21106, 318, 281, 12064, 326] ... [24993, 318, 366, 6726, 1911]
targets[0]: [318, 281, 12064, 326, 8477] ...  [318, 366, 6726, 1911, 50256]

Input tokens in entire batch:  148
Target tokens in entire batch: 148


> **Masking note:** `custom_collate_fn` replaces only *padding* tokens with `ignore_index` (-100). Prompt/instruction tokens are **not** masked, so they **currently contribute to the loss** along with the response. To train on the response only, also mask the prompt-token positions.

In [15]:
from functools import partial


pad_token_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})


customized_colate_fn = partial(custom_collate_fn,
                               device=device,
                               allowed_max_length=1024,
                               pad_token_id=pad_token_id[0]) 

# error if pad_token_id = [50256] instead of pad_token_id = 50256 since function already wraps it in brackets.

### DataLoader

In [16]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size  = 8
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    collate_fn=customized_colate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
    )

# Inspect a single example batch (not every batch) - shape differs due to shuffling
inputs, targets = next(iter(train_loader))
print(f"inputs.shape:  {inputs.shape}\ntargets.shape: {targets.shape}\n")

print(f"inputs[0]:  {inputs[0][:5].tolist()} ... {inputs[0][-5:].tolist()}")
print(f"targets[0]: {targets[0][:5].tolist()} ...  {targets[0][-5:].tolist()}")

inputs.shape:  torch.Size([8, 68])
targets.shape: torch.Size([8, 68])

inputs[0]:  [21106, 318, 281, 12064, 326] ... [50256, 50256, 50256, 50256, 50256]
targets[0]: [318, 281, 12064, 326, 8477] ...  [-100, -100, -100, -100, -100]


In [17]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    collate_fn=customized_colate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
    )

print(len(val_loader))

7


In [18]:
test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    collate_fn=customized_colate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
    )

print(len(test_loader))

14


### generate response before fine-tuning

In [8]:
from reference.generate import generate
from reference.model_io import text_to_token_ids, token_ids_to_text

torch.manual_seed(123)
sample_input = format_input(val_data[0])
token_ids = generate(model, 
idx = text_to_token_ids(sample_input, tokenizer).to(device), 
max_new_tokens = 35,
context_length = gpt2_BASE_CONFIG['context_length'],
eos_id = tokenizer.eot_token)

generated_text = token_ids_to_text(token_ids, tokenizer)
response_text = generated_text[len(sample_input):].strip()
print(f"{response_text}")

### Response:
The meal is cooked everyday by the chef.


## 3 · Loss / objective  ← differs
TODO: next-token cross-entropy; prompt tokens currently count (see masking note).

In [ ]:
# Loss before fine-tuning
from reference.losses import calc_loss_loader



## 4 · Generate / predict
TODO: generate a response; capture a baseline before training.

In [ ]:

torch.manual_seed(123)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training Loss before Fine-tuning:",   train_loss)
print("Validation Loss before Fine-tuning:", val_loss)

## 5 · Train

In [ ]:
%%time
from reference.train import train_model_simple

torch.manual_seed(123)
optimizer = torch.optim.AdamW(
    params=model.parameters(),
    lr=0.00005,
    weight_decay=0.1
)
num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, optimizer, device, tokenizer,
    train_loader, val_loader,
    eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), num_epoch=num_epochs
)


### Plot Losses

In [ ]:
from reference.plot import plot_losses
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)


## 6 · Save, load & resume training

**To download/save trained model from runpod**

    After running the Save cell:
    1. Open the Jupyter URL from setup.ipynb in your browser.
    2. Find Training/runs/instruction/002_finetuned/ → right-click the newest .pth → Download. Docs
    3. Move it into the matching folder on your Mac.

#### Save - model & optimizer 

In [ ]:
from pathlib import Path
from datetime import datetime

folder = Path("runs/instruction/002_finetuned")
folder.mkdir(parents=True, exist_ok=True)

checkpoint = {"model": model.state_dict(), "optimizer": optimizer.state_dict()}
torch.save(checkpoint, folder / f"checkpoint_{datetime.now():%Y%m%d_%H%M%S_%f}.pth")

#### Load - model & optimizer

    **Only turn on if starting form this cell**

In [1]:
# import json
# import torch
# from tqdm import tqdm
# from pathlib import Path
# from reference.generate import generate
# from reference.models.gpt2 import GPT2Model
# from reference.configs import gpt2_BASE_CONFIG, gpt2_model_configs
# from reference.model_io import text_to_token_ids, token_ids_to_text

# device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
# CHOOSE_MODEL = "gpt2-medium (355M)" # gpt2-small (124M), gpt2-medium (355M), gpt2-large (774M), gpt2-xl (1558M)
# model_config = {**gpt2_BASE_CONFIG, **gpt2_model_configs[CHOOSE_MODEL]}

In [7]:

# Latest saved model checkpoint
latest = max(Path("runs/instruction/002_finetuned").glob("checkpoint_*.pth"))
checkpoint = torch.load(latest, map_location=device)

model = GPT2Model(model_config).to(device)
model.load_state_dict(checkpoint["model"])

optimizer = torch.optim.AdamW(model.parameters())
optimizer.load_state_dict(checkpoint["optimizer"])

# model.train();  # continue training; use model.eval() to only generate text

#### Resume Training

In [3]:

# model.train();  # continue training; use model.eval() to only generate text

# train_losses, val_losses, tokens_seen = train_model_simple(
#     model, optimizer, device, tokenizer,
#     train_loader, val_loader,
#     eval_freq=5, eval_iter=5,
#     start_context=format_input(val_data[0]), num_epoch=1
# )

## 7 · Evaluate & reflect  ← differs
TODO: score responses (optionally local Qwen); compare to baseline.

#### Extract model responses & save

In [9]:
import json
from tqdm import tqdm

def extract_response_and_save(path, file_name):
    model.eval()
    for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
        input_text = format_input(entry)
        token_ids = generate(
            model,
            idx=text_to_token_ids(input_text, tokenizer).to(device),
            max_new_tokens=256,
            context_length=gpt2_BASE_CONFIG["context_length"],
            eos_id=tokenizer.eot_token
        )

        generated_text = token_ids_to_text(token_ids, tokenizer)
        response_text = generated_text[len(input_text):].replace("### Response:", "").strip()
        test_data[i]["model_response"] = response_text

    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    with open(path / file_name, "w") as file:
        json.dump(test_data, file, indent=4)

extract_response_and_save("runs/instruction/002_finetuned", "responses.json")


  0%|          | 0/110 [00:00<?, ?it/s]

  1%|          | 1/110 [00:00<00:57,  1.91it/s]

  2%|▏         | 2/110 [00:01<01:10,  1.53it/s]

  3%|▎         | 3/110 [00:02<01:16,  1.41it/s]

  4%|▎         | 4/110 [00:02<01:08,  1.56it/s]

...

 97%|█████████▋| 107/110 [01:21<00:10,  3.41s/it]

 98%|█████████▊| 108/110 [01:22<00:05,  2.50s/it]

 99%|█████████▉| 109/110 [01:22<00:01,  1.87s/it]

100%|██████████| 110/110 [01:23<00:00,  1.48s/it]

100%|██████████| 110/110 [01:23<00:00,  1.32it/s]

#### Evaluate using Local Model

In [10]:

from reference.evals import query_model
from tqdm import tqdm

def generate_model_scores(data, judge_model, think=None):
    scores = []
    for entry in tqdm(data, desc="Scoring"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry['model_response']}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        reply = query_model(prompt, model=judge_model, think=think)
        try:
            scores.append(int(reply))
        except ValueError:
            print(f"Skipped invalid score: {reply!r}")
    return scores

scores = generate_model_scores(test_data, "qwen3.8:27b-mlx", think=False)

print(f"Scored: {len(scores)}/{len(test_data)}")
if scores:
    print(f"Average score: {sum(scores) / len(scores):.2f}/100")


Scoring:   0%|          | 0/110 [00:00<?, ?it/s]

Scoring:   1%|          | 1/110 [00:31<57:00, 31.38s/it]

Scoring:   2%|▏         | 2/110 [00:33<25:43, 14.29s/it]

Scoring:   3%|▎         | 3/110 [00:37<16:53,  9.47s/it]

Scoring:   4%|▎         | 4/110 [00:39<11:17,  6.39s/it]

...

Scoring:  98%|█████████▊| 108/110 [03:12<00:03,  1.77s/it]

Scoring:  99%|█████████▉| 109/110 [03:13<00:01,  1.60s/it]

Scoring: 100%|██████████| 110/110 [03:15<00:00,  1.48s/it]

Scoring: 100%|██████████| 110/110 [03:15<00:00,  1.77s/it]

Scored: 110/110
Average score: 31.14/100
